# 3. Two sources, one question

**Going further.** Assumes pandas, and that you have seen a regression before.

One source answers one kind of question. Interesting questions usually need
two, and the joining is where the actual work is. This notebook is mostly about
the join, because that is where results quietly go wrong.

The question:

> *Do states with higher median household income have lower unemployment, and
> is that just picking up something about state size?*

Census knows about income by state. FRED knows about unemployment by state.
Neither knows about the other.

In [ ]:
import os
from pathlib import Path

env = Path(".env")
if env.exists():
    for line in env.read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()

import census_loader as cl
import fred_loader as fl
import pandas as pd

## Find the variable instead of knowing it

The Census API names things like `B19013_001E`. Nobody holds those. Search in
words, which is the whole reason this library exists.

In [ ]:
cl.search("median household income")

In [ ]:
# `info` tells you what a series actually measures before you build on it.
# Reading this is a thirty second habit that prevents a whole class of
# confident wrong answers, the kind where the number is fine and the
# INTERPRETATION was never checked.
cl.info("median_household_income")

In [ ]:
income = cl.pull_census(
    ["median_household_income", "total_population"],
    geography="state",
    year=2022,
)
income.head()

## The other half

FRED carries state unemployment as one series per state, coded `XXUR`: `CAUR`
for California, `NYUR` for New York. Build the codes rather than typing fifty.

In [ ]:
STATES = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
]

codes = [f"{s}UR" for s in STATES]
unemployment = fl.pull_fred(codes, start="2022-01-01", end="2022-12-31")
print(f"{unemployment.shape[1]} series, {len(unemployment)} months")
unemployment.iloc[:3, :5]

## Here is the join, and here is the decision inside it

Census income is **one number for the year**. FRED unemployment is **twelve
monthly numbers**. They cannot be joined until they describe the same period.

You have to collapse twelve numbers into one, and *how* you collapse is a claim:

- **mean of the twelve** says "the typical month that year"
- **December only** says "where it ended up"
- **max** says "the worst it got"

For "what was unemployment like in this state in 2022", the annual mean is the
honest match to an annual income figure. The others answer different questions
and would be defensible in a write-up that said so.

**The failure here is never picking wrong. It is picking without noticing.**

In [ ]:
annual = unemployment.mean().rename("unemployment").to_frame()
annual.index = [code[:2] for code in annual.index]   # CAUR -> CA
annual.index.name = "state"
annual.head()

## Join, and count what you lost

The two sources also disagree about how to *name* a state: Census uses full
names or FIPS codes, FRED uses two letters. Something has to translate.

**Always count rows before and after a join.** A silent drop from 50 to 31 is
the single most common way an analysis ends up describing a different
population than the one you think you are describing.

In [ ]:
# Look at what the Census frame actually gives you to join ON, rather than
# assuming a column name.
print(income.columns.tolist())
income.head(3)

In [ ]:
ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT",
    "Delaware": "DE", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME",
    "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI",
    "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO",
    "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND",
    "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR",
    "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
}

name_col = next(c for c in income.columns if income[c].dtype == object)
left = income.copy()
left["state"] = left[name_col].map(ABBREV)

# WHO FAILED TO MATCH, by name. Puerto Rico and DC are the usual answer and
# are legitimately not states; anything else is a bug in the mapping.
unmatched = left.loc[left["state"].isna(), name_col].tolist()
print(f"did not map: {unmatched}")

merged = left.dropna(subset=["state"]).merge(annual, on="state", how="inner")
print(f"census {len(income)} + fred {len(annual)} -> joined {len(merged)}")
merged.head()

## Now the question, with its control

The original question had two halves: does income go with unemployment, and is
that just state size. Population is the control that answers the second half.

In [ ]:
import numpy as np
import statsmodels.api as sm
from otter import Pond

model_df = merged.copy()
income_col = "median_household_income"
pop_col = "total_population"

# Thousands of dollars, so the coefficient reads in a unit a person has an
# intuition about. Log population because state sizes span three orders of
# magnitude and the raw number would let California decide everything.
model_df["income_k"] = model_df[income_col] / 1000
model_df["log_pop"] = np.log(model_df[pop_col])
model_df = model_df.dropna(subset=["income_k", "log_pop", "unemployment"])

pond = Pond(model_df)
pond.set_dependent("unemployment")
pond.add_independents("income_k")
pond.add_controls("log_pop")
pond.get_spec()

In [ ]:
X = sm.add_constant(pond.get_X())
y = pond.get_y()
print(sm.OLS(y, X).fit().summary())

## What to take from it

Read `income_k` in its units: a coefficient of -0.02 means ten thousand dollars
more median income goes with 0.2 points lower unemployment.

**Then run it again without the control** and compare. If the income
coefficient barely moves, income was not standing in for size. If it moves a
lot, it partly was, and the first version was telling you about population
wearing income's clothes.

**And the limit worth saying out loud:** fifty rows, one year. This is a
cross-section of a single snapshot. It cannot tell you what happens to a state
when its income rises, only that richer states differed from poorer ones in
2022. Those are different sentences and only one of them is supported.

## The part that generalises

Almost nothing above was statistics:

1. Two sources described time differently, and collapsing one was a choice
2. Two sources named the same thing differently, and something had to translate
3. The join lost rows, and you checked which by name rather than trusting a count
4. Units were chosen so the coefficient could be read aloud

That is the shape of nearly every real analysis. The model at the end is the
short part.

**Next:** [`04-is-the-difference-real.ipynb`](04-is-the-difference-real.ipynb),
on deciding how much of an answer is signal.